# Five exercises: what the ARC constructions actually do

About ten minutes. Each exercise asks you to **commit to an answer before running**
the check. The point is not to be right — it is that the places where your
prediction and the code disagree are exactly the design decisions worth arguing
about.

Fill in `answer = ...`, run the cell, then run the check below it.

In [ ]:
import numpy as np
import pandas as pd

import featuregraph as fg
from featuregraph.behaviors.composition import BlockComposition, resolve_layout
from featuregraph.behaviors.regions import RegionObjects, edit_alignment

PAIR = ["task_id", "pair_type", "pair_index"]
GRID = [*PAIR, "grid_role"]


def cells_from(grids, task_id="exercise"):
    """Cell-level observations from {role: grid} for one pair."""
    records = []
    for grid_role, grid in grids.items():
        grid = np.asarray(grid, dtype=int)
        records += [
            {
                "task_id": task_id,
                "pair_type": "train",
                "pair_index": 0,
                "grid_role": grid_role,
                "row": row,
                "column": column,
                "color": int(grid[row, column]),
            }
            for row, column in np.ndindex(grid.shape)
        ]
    return pd.DataFrame.from_records(records)


def check(label, answer, actual, lesson):
    verdict = "correct" if answer == actual else f"not quite — it is {actual}"
    print(f"{label}: you said {answer} — {verdict}\n")
    print(lesson)

---
## 1. The declaration makes the objects

This grid has three non-background cells, arranged on a diagonal:

```text
1 0 0
0 1 0
0 0 2
```

You are going to build region objects from it four times, varying two declared
parameters: `definition` (`uniform_color` groups adjacent cells of the same colour;
`foreground_mask` groups adjacent non-background cells whatever their colours) and
`connectivity` (4 excludes diagonal neighbours, 8 includes them).

**How many regions does each of the four combinations produce?** Background is
excluded. Write four numbers.

In [ ]:
# uniform_color/4, uniform_color/8, foreground_mask/4, foreground_mask/8
answer = [None, None, None, None]

In [ ]:
diagonal = cells_from({"input": [[1, 0, 0], [0, 1, 0], [0, 0, 2]]})

actual = []
for definition in ("uniform_color", "foreground_mask"):
    for connectivity in (4, 8):
        builder = RegionObjects(
            group=GRID,
            definition=definition,
            connectivity=connectivity,
            include_background=False,
        )
        objects = builder.summarize(builder.fit_transform(diagonal))
        actual.append(objects.count)

check(
    "Exercise 1",
    answer,
    actual,
    "Same observations, four different sets of objects. Nothing was detected —\n"
    "the declaration decided what an object is. This is why connectivity and\n"
    "definition are recorded in objects.construction rather than hard-coded.",
)

---
## 2. Ambiguity is data, not an error

A 2×2 grid of a single colour, whose output block is that same grid:

```text
input        output
3 3          3 3
3 3          3 3
```

`BlockComposition` checks seven declared operators against that block: `copy`,
`flip_horizontal`, `flip_vertical`, `rotate_90`, `rotate_180`, `rotate_270`,
`background`.

**How many of the seven describe this block exactly?**

In [ ]:
answer = None

In [ ]:
uniform = cells_from({"input": [[3, 3], [3, 3]], "output": [[3, 3], [3, 3]]})

builder = BlockComposition(group=PAIR)
objects = builder.summarize(builder.fit_transform(uniform))
block = objects.to_pandas().iloc[0]

print("candidates:", sorted(block["candidates"]))
check(
    "Exercise 2",
    answer,
    int(block["candidate_count"]),
    "A symmetric grid cannot distinguish these operators, so the block keeps all\n"
    "of them. A solver has to pick one and is silently wrong later, or raise.\n"
    "Here the ambiguity is a value in a column you can query.",
)

---
## 3. The grouping is the hypothesis

Task `007bbfb7` tiles a copy of the input wherever an input cell is non-background,
and background elsewhere. Its output is 9×9 from a 3×3 input, so there are nine
blocks.

`resolve_layout` intersects each block's candidate set across the demonstration
pairs. Grouping **by block coordinate** asks: does each position always take the
same operator?

**How many of the nine block coordinates resolve to exactly one operator?**

In [ ]:
answer = None

In [ ]:
fractal = fg.datasets.arc_agi("007bbfb7", split="training")
builder = BlockComposition(group=PAIR)
objects = builder.summarize(builder.fit_transform(fractal))
demonstrations = objects.query().where(pair_type="train").collect()

by_position = resolve_layout(demonstrations, by=("block_row", "block_column"))
print(by_position[["block_row", "block_column", "candidate_count"]].to_string(index=False))

check(
    "Exercise 3",
    answer,
    int(by_position["is_determined"].sum()),
    "Zero. This task's layout depends on the input contents, so asking about\n"
    "fixed positions asks a question the task does not answer.",
)

Now the same objects, grouped by the **state of the input cell** each block
corresponds to, instead of by position. Nothing is recomputed — this is the same
table with a different `GROUP BY`.

**What does each state resolve to?**

In [ ]:
by_state = resolve_layout(demonstrations, by=("block_state",))
print(by_state[["block_state", "candidate_count", "operator"]].to_string(index=False))

print(
    "\nThe two 'solver families' in featuregraph/utils/_arc_agi.py are these two\n"
    "lines. Not two algorithms — two groupings of one object table. A third\n"
    "hypothesis would be a third grouping, not a third solver."
)

---
## 4. Failure has a size

Task `0692e18c` is a tiling task the seven-operator vocabulary does **not** fully
describe. Its demonstrations contain 27 blocks in total.

The old code path raised `ValueError` on the first block it could not match, so the
only available answer was "unsupported".

**How many of the 27 blocks does the vocabulary fail to describe?** Any number from
0 to 27.

In [ ]:
answer = None

In [ ]:
partial = fg.datasets.arc_agi("0692e18c", split="training")
builder = BlockComposition(group=PAIR)
objects = builder.summarize(builder.fit_transform(partial))
train = objects.query().where(pair_type="train").collect()

print(f"blocks: {len(train)}   determined: {int(train['is_determined'].sum())}   "
      f"unmatched: {int(train['is_unmatched'].sum())}")

check(
    "Exercise 4",
    answer,
    int(train["is_unmatched"].sum()),
    "The task is 15/27 describable. 'Unsupported' was true but threw away the\n"
    "resolution: which blocks, how many, and whether the gap is one stubborn\n"
    "corner or the whole grid.",
)

---
## 5. Two questions that look like one

Task `1acc24af` is a same-shape task — output dimensions equal input dimensions —
so block composition says nothing about it. Region objects do.

Answer **both**, they are independent:

- **(a)** Of the edits between input and output, what fraction lie entirely inside a
  single input region? (0.0 to 1.0)
- **(b)** Is the output colour of a region a function of its input colour, size,
  size rank, or bounding box? (`True` / `False`)

In [ ]:
answer_a = None  # fraction, e.g. 0.5
answer_b = None  # True or False

In [ ]:
task = fg.datasets.arc_agi("1acc24af", split="training")
edits = edit_alignment(task)

probe = pd.read_csv("../artifacts/arc/region_probe.csv")
row = probe[(probe.task_id == "1acc24af") & (probe.connectivity == 4)].iloc[0]
predictable = bool(
    row[
        [
            "colour_determines_output",
            "size_determines_output",
            "size_rank_determines_output",
            "bbox_determines_output",
        ]
    ].any()
)

print(f"edit components: {len(edits)}")
check("Exercise 5a", answer_a, edits["is_aligned"].mean(), "")
check(
    "Exercise 5b",
    answer_b,
    predictable,
    "Every edit respects region boundaries, and no simple region property\n"
    "predicts what the edit does. Regions are the right object boundary AND the\n"
    "edits are not predictable — both true at once.",
)

---
## What the five add up to

1. Objects come from a **declaration**, and the declaration is recorded.
2. Ambiguity is **retained as data** instead of resolved or raised.
3. A hypothesis class is a **grouping** of the object table, not a separate solver.
4. Failure has a **size and a location**, not just a verdict.
5. "Is this the right object boundary" and "can I predict the change" are
   **different questions** with different answers.

Point 5 is the one carrying the argument that this is a representation project
rather than a solver project — and it is the one I would most want you to disagree
with me about if you do.

Full context: [`artifacts/arc/README.md`](../artifacts/arc/README.md) for the block
study, [`artifacts/arc/region_objects_scope.md`](../artifacts/arc/region_objects_scope.md)
for the region scope and what is deliberately not being built.